## 2.4 文本预处理 - Embedding 的本质与文本如何真正变成 RNN 输入

#### 1、为什么这一小节非常重要

##### 1.1 前面我们只是把文本“编号化”，这一节才真正进入“向量化”
前面几小节我们已经建立了这样一条链路：

原始文本  
$\rightarrow$ 分词  
$\rightarrow$ token  
$\rightarrow$ 词表 `vocabulary`  
$\rightarrow$ token 对应 id  
$\rightarrow$ one-hot 的思想  
$\rightarrow$ 发现 one-hot 不够好

到这里为止，我们已经知道：

- 文本不能直接输入 RNN
- token 先要变成 id
- id 只是编号，不代表语义
- one-hot 虽然比 id 更合理，但仍然存在高维、稀疏、无语义相似性等问题

那么接下来就自然要回答一个关键问题：

既然 id 不够，one-hot 也不够，那么文本到底应该以什么形式真正输入 RNN？

答案就是：

Embedding 向量序列。

所以这一小节非常关键，因为它真正把“自然语言”连接到了“RNN 输入张量”。

##### 1.2 这一小节是文本输入模型的真正入口
从学习顺序上说，前面的内容更像是在铺路：

- 什么是 token
- 什么是 vocabulary
- token 如何转 id
- 为什么需要比 one-hot 更好的表示

而这一节开始，我们终于进入：

文本如何被表示成模型可以直接计算的向量

也就是说，从这一节开始，文本不再只是概念层面的“字符串处理”，而是正式变成了神经网络输入。

##### 1.3 这一节会把前面所有知识串起来
这一小节会把前面的所有内容真正整合到一起：

- 为什么先需要词表
- 为什么先有 id
- 为什么 Embedding 能接在 id 后面
- 为什么 Embedding 的输出正好适合输入 RNN
- 为什么 RNN 的 `input_size` 往往等于 `embedding_dim`

所以这一节既是一个新知识点，也是前面几小节的汇总和落地。

#### 2、Embedding 到底是什么

##### 2.1 Embedding 的最直观理解
Embedding 可以先用一句最简单的话来理解：

Embedding 就是把每个 token 的编号，映射成一个低维、稠密、可训练的向量。

例如：

`love` 这个词的 id 是 `3`

经过 Embedding 之后，它不再只是数字 `3`，而会变成类似这样的向量：

```python
[0.12, -0.37, 0.85, 0.44]
```

这个向量才是后面真正送进 RNN 的输入。

所以你可以把 Embedding 理解成：

“把单词编号翻译成向量表示”的一层。

##### 2.2 Embedding 不是简单换个数字，而是换成“特征向量”
这一点非常重要。

id 只是身份编号，例如：

`love → 3`

但 Embedding 之后：

`love → [0.12, -0.37, 0.85, 0.44]`

你会发现，两者本质完全不同：

- `3` 只是编号
- 后面的向量是一组连续数值特征

也就是说，Embedding 不是单纯把一个数字换成另一个数字，而是把一个离散符号变成一个连续向量表示。

这才是神经网络真正擅长处理的输入形式。

##### 2.3 Embedding 的关键词：低维、稠密、可训练
理解 Embedding 时，最好牢牢记住这 3 个关键词：

**（1）低维**

相对于 one-hot 那种维度等于词表大小的超长向量，Embedding 通常维度更低，例如：

- `32` 维
- `64` 维
- `128` 维
- `300` 维

**（2）稠密**

Embedding 向量中的每个位置通常都是实数，不再像 one-hot 那样几乎全是 `0`。

**（3）可训练**

Embedding 不是固定死的，它会随着训练不断更新，让模型逐渐学出更适合当前任务的词向量。

#### 3、Embedding 的本质是什么

##### 3.1 本质上它是一个“查表层”
Embedding 最适合初学者的理解方式，就是：

它本质上像一个查表层。

假设：

- 词表大小是 `10000`
- Embedding 维度是 `128`

那么 Embedding 层内部其实可以看成有一张大表：

`10000` 行，`128` 列

这张表中的：

- 每一行对应一个 token
- 每一行就是这个 token 的向量表示

如果：

`love` 的 id `= 3`

那么 Embedding 层就会去这张表里找第 `3` 行，把那一整行取出来，作为 `love` 的词向量。

所以你可以理解为：

Embedding(id) = 取出“第 id 行向量”

##### 3.2 从数学上看，它是一个矩阵
如果我们用数学符号来表示：

- 词表大小 $= V$
- Embedding 维度 $= d$

那么 Embedding 层可以看成一个矩阵：

$E \in \mathbb{R}^{V \times d}$

其中：

- $V$ 表示词表里有多少个 token
- $d$ 表示每个 token 用多少维向量表示

例如：

- $V = 10000$
- $d = 128$

那么 Embedding 矩阵就是：

$10000 \times 128$

这表示：

- 总共有 `10000` 个 token
- 每个 token 都有一个 `128` 维向量

##### 3.3 每个 token 的 id 就决定了要取矩阵的哪一行
例如：

`love` 的 id `= 3`

那么它对应的向量就是：

$E[3]$

如果：

`AI` 的 id `= 4`

那它对应的向量就是：

$E[4]$

所以 Embedding 的核心操作其实很简单：

- 输入一个 id
- 输出这一行对应的向量

也正因为如此，前面为什么一定要先建立词表、先做 token 到 id 的映射，现在你就能看明白了。

因为如果没有 id，Embedding 根本不知道该查哪一行。

#### 4、Embedding 向量是怎么来的

##### 4.1 一开始通常是随机初始化的
在训练刚开始时，Embedding 矩阵中的每一行通常只是随机数。

例如：

`love` 的向量一开始可能只是随机生成的：

```python
[0.07, -0.15, 0.03, 0.29]
```

它并不是一开始就“懂语义”。

##### 4.2 在训练过程中不断更新
随着模型训练，损失函数会进行反向传播，Embedding 矩阵中的参数也会一起更新。

于是模型会逐渐学到：

- 哪些词应该更接近
- 哪些词对当前任务更重要
- 哪些维度更有帮助

所以 Embedding 其实也是模型参数的一部分。

##### 4.3 最终学出来的是“对当前任务有用的词向量”
这一点很重要。

Embedding 不是抽象地学“词典定义”，而是学：

对于当前任务最有用的向量表示

例如：

- 做情感分类时，模型可能更关注正负情绪相关的词义差异
- 做主题分类时，模型可能更关注话题相关的词义差异

所以 Embedding 是任务相关、数据驱动地学出来的。


#### 5、为什么 Embedding 比 one-hot 更适合文本表示

##### 5.1 它把高维稀疏向量变成了低维稠密向量
假设词表大小是 `10000`。

如果用 one-hot，那么每个 token 都要表示成一个长度为 `10000` 的向量，例如：

```python
[0, 0, 0, 0, 1, 0, 0, ...]
```

这会带来两个问题：

- 维度太高
- 大部分位置都是 `0`，非常稀疏

而如果用 Embedding，可以把它压缩成一个长度为 `128` 的稠密向量，例如：

```python
[0.12, -0.37, 0.85, 0.44, ...]
```

这样表达会更紧凑，计算也更高效。

##### 5.2 它更容易表达词与词之间的相似性
这是 Embedding 相比 one-hot 最重要的优势。

在 one-hot 中：

`good`、`great`、`excellent` 之间的向量一样“远”，因为它们只是不同位置被点亮。

但在 Embedding 中，如果这些词经常出现在相似上下文中，模型就可能把它们学成相近的向量。

例如：

```python
good  -> [0.81, 0.22, -0.11, 0.49]
great -> [0.79, 0.25, -0.09, 0.52]
```

你会发现它们可能会比较接近。

这就意味着：

Embedding 能在向量空间中表达语义相似性。

##### 5.3 它更适合神经网络学习
神经网络特别擅长处理连续空间中的数值特征。

而 Embedding 正好把原本离散的单词，变成了连续向量。

这样后面的 RNN 在处理这些输入时，就不再只是处理“编号”，而是在处理“可学习的特征向量”。


#### 6、一个词经过 Embedding 之后会发生什么

##### 6.1 输入前：它只是一个整数 id
例如：

`love → 3`

在进入 Embedding 层之前，它还只是一个离散编号。

##### 6.2 经过 Embedding：它变成一个向量
假设 `embedding_dim = 4`

那么经过 Embedding 之后，可能变成：

```python
love -> [0.12, -0.37, 0.85, 0.44]
```

这时，这个词就不再是单独的编号，而是一个 `4` 维向量。

##### 6.3 这个向量就是后面 RNN 某个时间步的输入
如果句子是：

`I / love / AI`

那么：

- 第 `1` 个时间步输入 `I` 的向量
- 第 `2` 个时间步输入 `love` 的向量
- 第 `3` 个时间步输入 `AI` 的向量

所以在 RNN 的视角里，它看到的并不是单词本身，而是一串按时间顺序排列的向量。

#### 7、一句话经过 Embedding 后会变成什么

##### 7.1 先从一句最简单的话开始
句子：

`I love AI`

假设词表如下：

- `<PAD> → 0`
- `<UNK> → 1`
- `I → 2`
- `love → 3`
- `AI → 4`

##### 7.2 先转成 id 序列
`I / love / AI`  
$\rightarrow$  
`2 / 3 / 4`

如果需要补齐到长度 `5`，那么可能是：

`2 / 3 / 4 / 0 / 0`

##### 7.3 再经过 Embedding
假设 `embedding_dim = 4`

那么：

```python
2 -> [0.20, -0.10, 0.60, 0.30]
3 -> [0.12, -0.37, 0.85, 0.44]
4 -> [0.55, 0.91, -0.08, 0.17]
0 -> [0.00, 0.00, 0.00, 0.00]
0 -> [0.00, 0.00, 0.00, 0.00]
```

于是整句话就会变成一个向量序列：

```python
[
    [0.20, -0.10, 0.60, 0.30],
    [0.12, -0.37, 0.85, 0.44],
    [0.55, 0.91, -0.08, 0.17],
    [0.00, 0.00, 0.00, 0.00],
    [0.00, 0.00, 0.00, 0.00]
]
```

这时，这句话终于变成了真正可以送入 RNN 的输入形式。

#### 8、Embedding 输出后的张量形状怎么理解

这一部分非常关键，因为它会直接连接到 RNN 输入维度。

##### 8.1 单个句子的情况
假设：

- 单个序列长度 `seq_len = 5`
- Embedding 维度 `embedding_dim = 4`

那么一个句子经过 Embedding 后，可以表示成一个形状为：

$5 \times 4$

的矩阵。

为什么是 $5 \times 4$？

因为：

- 一共有 `5` 个 token，也就是 `5` 个时间步
- 每个 token 都变成了一个 `4` 维向量

所以：

- 行数 = 时间步个数
- 列数 = 每一步输入向量的维度

##### 8.2 加上 batch 之后
假设：

- `batch_size = 32`
- `seq_len = 20`
- `embedding_dim = 128`

那么一整个 batch 的输入形状通常就是：

$32 \times 20 \times 128$

这三个维度分别表示：

- `32`：一个 batch 中有 `32` 个样本
- `20`：每个样本统一成 `20` 个 token，统一长度
- `128`：每个 token 被表示成 `128` 维向量

这就是 NLP 任务中最常见的 RNN 输入张量形式。

#### 9、PyTorch 中的 Embedding 是怎么体现的

##### 9.1 最常见的写法
在 PyTorch 中，Embedding 层通常写成：

```python
nn.Embedding(num_embeddings, embedding_dim)
```

其中：

- `num_embeddings` 就是词表大小
- `embedding_dim` 就是词向量维度

##### 9.2 一个直观例子
假设：

- 词表大小 = `10000`
- `embedding_dim = 128`

那么可以写成：

```python
embedding = nn.Embedding(10000, 128)
```

这表示：

建立一个 `10000 × 128` 的 Embedding 矩阵

以后输入一个 token id，都会返回对应的 `128` 维向量。

##### 9.3 输入和输出的形状变化
如果输入是：

```python
[2, 3, 4, 0, 0]
```

形状是：`5`

那么经过 Embedding 后，输出就变成：

$5 \times 128$

如果输入是一个 batch（`batch_size = 32`）：

$32 \times 20$

那么经过 Embedding 后，输出就会变成：

$32 \times 20 \times 128$

这正好可以接入 `batch_first=True` 的 RNN。